# `aidp-streaming-kafka` live test — SASL_SSL OAuthBearer
**Live-test row 13.** Requires custom OAuthBearer callback handler JAR pre-attached to the cluster.


In [ ]:
import sys, os
# Adjust this if you've uploaded the plugin scripts/ dir elsewhere.
sys.path.insert(0, '/Workspace/Shared/oracle_ai_data_platform_connectors/scripts')


In [ ]:
from oracle_ai_data_platform_connectors.streaming import (
    bootstrap_for_region, build_kafka_options_oauthbearer, validate_checkpoint_path,
)

bootstrap = bootstrap_for_region(os.environ['OCI_REGION'])
opts = build_kafka_options_oauthbearer(
    bootstrap_servers=bootstrap,
    token_endpoint_url=os.environ['OCI_OAUTH_TOKEN_URL'],
    callback_handler_class=os.environ['KAFKA_OAUTH_CALLBACK_CLASS'],
    topic=os.environ['KAFKA_TOPIC'],
)


In [ ]:
checkpoint = validate_checkpoint_path(os.environ['KAFKA_CHECKPOINT_VOLUME'])
raw = spark.readStream.format('kafka').options(**opts).load()
query = raw.writeStream.format('memory').queryName('kafka_oauth_test').option('checkpointLocation', checkpoint).start()
query.awaitTermination(timeout=60)
df = spark.sql('SELECT * FROM kafka_oauth_test')
print('input rows in last batch:', query.lastProgress.get('numInputRows'))


In [ ]:
# Live-test driver parses this final cell's stdout for the JSON summary.
import json, time
summary = {
    'connector': 'aidp-streaming-kafka',
    'auth': 'oauthbearer',
    'rows': int(df.count()),
    'schema': sorted([f.name for f in df.schema.fields]),
    'timestamp_utc': int(time.time()),
}
print('AIDP_LIVE_TEST_RESULT_BEGIN')
print(json.dumps(summary, indent=2))
print('AIDP_LIVE_TEST_RESULT_END')
